In [2]:
import time
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import precision_score, recall_score, f1_score

# ==============================================================================
# 0. CONFIGURATION ET CHARGEMENT DU MODÈLE (LIGHTNING AI GPU)
# ==============================================================================
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct" # Modifie si tu utilises un autre nom de base
FINETUNED_PATH = "./loria_llama_attack_model/final_model"   # Ton dossier avec l'adaptateur LoRA
BENCHMARK_FILE = "benchmark_attack_eval.json"
OUTPUT_FILE = "results_eval_attacks.json"

print("="*70)
print(f"🚀 INITIALISATION DE L'ÉVALUATION CYBER (MODE GPU)")
print("="*70)

# --- DÉTECTION DU MATÉRIEL ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Matériel détecté : {device.upper()}")
if device == "cuda":
    print(f"🎮 Carte Graphique : {torch.cuda.get_device_name(0)}")

# --- CHARGEMENT DU TOKENIZER ---
print("⏳ Chargement du Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- CHARGEMENT DU MODÈLE (Standard HuggingFace, PAS Unsloth) ---
print("⏳ Chargement du Modèle de Base (en float16 pour optimiser la VRAM)...")
model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto" # Lightning AI va gérer le placement GPU automatiquement
)

print("⏳ Fusion avec les poids LoRA (loria_llama_attack_model)...")
model = PeftModel.from_pretrained(model_base, FINETUNED_PATH)
model.eval() # Passage officiel en mode inférence
print("✅ Modèle expert prêt pour l'analyse !")

# ==============================================================================
# 1. LE FILTRE SÉMANTIQUE & MAPPING (Spécial Cyberattaques)
# ==============================================================================

# Convertit le joli texte du JSON de test en label strict
MAPPING_VERITE_TERRAIN = {
    "Attaque (DDoS Volumétrique)": "ddos",
    "Attaque (Brute Force)": "brute_force",
    "Attaque (SQL Injection)": "sql_injection",
    "Attaque (Cryptojacking)": "cryptojacking",
    "Attaque (API Scanning / Reconnaissance)": "api_scan",
    "Attaque (Ransomware / Chiffrement)": "ransomware",
    "Attaque (SSRF - Server-Side Request Forgery)": "ssrf",
    "Attaque (Privilege Escalation)": "privilege_escalation"
}

def normaliser_attaque_ia(texte):
    """Traduit la prédiction brute de l'IA en label strict pour le calcul F1"""
    texte = str(texte).lower().strip()
    if "ddos" in texte or "volum" in texte: return "ddos"
    if "brute" in texte or "force" in texte: return "brute_force"
    if "sql" in texte or "injection" in texte: return "sql_injection"
    if "crypto" in texte or "xmrig" in texte: return "cryptojacking"
    if "scan" in texte or "reconnaissance" in texte: return "api_scan"
    if "ransomware" in texte or "chiffrement" in texte: return "ransomware"
    if "ssrf" in texte or "forgery" in texte: return "ssrf"
    if "privilege" in texte or "escalation" in texte or "admin" in texte: return "privilege_escalation"
    return "autre"

# ==============================================================================
# 2. LE PARSEUR ROBUSTE
# ==============================================================================
def parser_reponse_ia(reponse_brute):
    """Extrait le JSON généré par Llama 3.2 ou fouille le texte en cas de problème."""
    match = re.search(r'\{.*?\}', reponse_brute, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            svc = data.get("service_defaillant", data.get("affected_service", "non_identifié"))
            typ = data.get("type_panne", data.get("anomaly_type", "non_identifié"))
            return str(svc).strip().lower(), str(typ).strip()
        except json.JSONDecodeError:
            pass 
            
    # Plan B si le JSON est cassé
    svc_trouve, typ_trouve = "non_identifié", "non_identifié"
    service_match = re.search(r'([a-zA-Z0-9_]+service|frontend|redis-cart)', reponse_brute, re.IGNORECASE)
    if service_match:
        svc_trouve = service_match.group(1).lower()
        
    typ_trouve = normaliser_attaque_ia(reponse_brute)
    return svc_trouve, typ_trouve

# ==============================================================================
# 3. LA FONCTION DE GÉNÉRATION (PROMPT COMPLET POUR GPU)
# ==============================================================================
def generer_reponse(donnees_triplet):
    # Sur GPU, on donne 150 jetons au modèle pour qu'il rédige la "raison"
    system_prompt = """Tu es un expert SOC en cybersécurité. Analyse cette télémétrie.
Réponds STRICTEMENT par un JSON valide avec cette structure exacte :
{"service_defaillant": "nom_du_service", "type_panne": "choix: [ddos, brute_force, sql_injection, cryptojacking, api_scan, ransomware, ssrf, privilege_escalation]", "raison": "explication brève"}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": donnees_triplet}
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024 
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150, 
            temperature=0.1,    
            pad_token_id=tokenizer.eos_token_id
        )

    reponse_brute = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return reponse_brute

# ==============================================================================
# 4. LA BOUCLE D'ÉVALUATION PRINCIPALE
# ==============================================================================
try:
    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)
except FileNotFoundError:
    print(f"❌ ERREUR : Le fichier {BENCHMARK_FILE} est introuvable. As-tu lancé le script de génération ?")
    exit()

print(f"\n🚀 LANCEMENT DES TESTS SUR {len(benchmark)} SCÉNARIOS...")
resultats = []
temps_total = 0

for i, cas in enumerate(benchmark):
    t_debut = time.time()
    
    # 1. Inférence
    reponse_brute = generer_reponse(cas["input"])
    
    duree_cas = time.time() - t_debut
    temps_total += duree_cas

    # 2. Parsing de la prédiction de l'IA
    service_predit, type_predit_brut = parser_reponse_ia(reponse_brute)
    type_predit_normalise = normaliser_attaque_ia(type_predit_brut)

    # 3. Récupération de la vérité terrain (Ground Truth) depuis le JSON
    service_attendu = str(cas["reponse_attendue"]["service_defaillant"]).lower().strip()
    type_attendu_texte = cas["reponse_attendue"]["type_panne"]
    type_attendu_normalise = MAPPING_VERITE_TERRAIN.get(type_attendu_texte, "inconnu")

    # 4. Comparaison
    svc_ok = (service_predit == service_attendu)
    type_ok = (type_predit_normalise == type_attendu_normalise)
    ok_total = svc_ok and type_ok

    resultats.append({
        "nom_cas": cas.get("nom_cas", f"Cas {i+1}"),
        "service_attendu": service_attendu,
        "service_predit": service_predit,
        "type_attendu": type_attendu_normalise,
        "type_predit": type_predit_normalise,
        "complet_ok": ok_total,
        "reponse_brute_ia": reponse_brute
    })

    # Affichage dynamique
    acc_courante = sum(1 for r in resultats if r["complet_ok"]) / (i + 1) * 100
    s_ic = "✅" if svc_ok else "❌"
    t_ic = "✅" if type_ok else "❌"

    if not ok_total:
        print(f"[{i+1:3d}/{len(benchmark)}] ⚠️ ERREUR | {duree_cas:.1f}s")
        print(f"   Attendu : {service_attendu} / {type_attendu_normalise}")
        print(f"   Obtenu  : {service_predit} / {type_predit_normalise}")
    else:
        print(f"[{i+1:3d}/{len(benchmark)}] {s_ic} svc={service_predit[:15]:<15} {t_ic} type={type_predit_normalise[:15]:<15} | acc={acc_courante:5.1f}% | {duree_cas:.1f}s")

print(f"\n✅ Évaluation terminée en {temps_total/60:.1f} minutes")

# ==============================================================================
# 5. SAUVEGARDE EN JSON ET MÉTRIQUES SCIENTIFIQUES
# ==============================================================================
try:
    # L'enregistrement strict demandé dans results_eval_attacks.json
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(resultats, f, ensure_ascii=False, indent=4)
    print(f"💾 Résultats détaillés sauvegardés dans '{OUTPUT_FILE}'")
except Exception as e:
    print(f"⚠️ Erreur lors de la sauvegarde : {e}")

y_true_services = [r["service_attendu"] for r in resultats]
y_pred_services = [r["service_predit"] for r in resultats]
y_true_types = [r["type_attendu"] for r in resultats]
y_pred_types = [r["type_predit"] for r in resultats]

print("\n" + "="*50)
print("📊 RÉSULTATS SCIENTIFIQUES GLOBAUX (CYBERATTAQUES)")
print("="*50)

# --- Métriques Services ---
prec_svc = precision_score(y_true_services, y_pred_services, average='macro', zero_division=0)
rec_svc  = recall_score(y_true_services, y_pred_services, average='macro', zero_division=0)
f1_svc   = f1_score(y_true_services, y_pred_services, average='macro', zero_division=0)

print("\n🎯 IDENTIFICATION DU SERVICE CIBLE (Localisation) :")
print(f"   Précision : {prec_svc:.4f}")
print(f"   Rappel    : {rec_svc:.4f}")
print(f"   F1-Score  : {f1_svc:.4f}")

# --- Métriques Cyberattaques ---
prec_typ = precision_score(y_true_types, y_pred_types, average='macro', zero_division=0)
rec_typ  = recall_score(y_true_types, y_pred_types, average='macro', zero_division=0)
f1_typ   = f1_score(y_true_types, y_pred_types, average='macro', zero_division=0)

print("\n🎯 CLASSIFICATION DE L'ATTAQUE (Vecteur de compromission) :")
print(f"   Précision : {prec_typ:.4f}")
print(f"   Rappel    : {rec_typ:.4f}")
print(f"   F1-Score  : {f1_typ:.4f}")

# --- Exact Match ---
exact_match_acc = (sum(1 for r in resultats if r["complet_ok"]) / len(resultats)) * 100
print(f"\n🏆 EXACT MATCH ACCURACY (Service ET Attaque corrects) : {exact_match_acc:.2f}%")
print("="*50)

🚀 INITIALISATION DE L'ÉVALUATION CYBER (MODE GPU)
⚡ Matériel détecté : CUDA
🎮 Carte Graphique : Tesla T4
⏳ Chargement du Tokenizer...
⏳ Chargement du Modèle de Base (en float16 pour optimiser la VRAM)...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

⏳ Fusion avec les poids LoRA (loria_llama_attack_model)...
✅ Modèle expert prêt pour l'analyse !

🚀 LANCEMENT DES TESTS SUR 160 SCÉNARIOS...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[  1/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 5.0s
[  2/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.8s
[  3/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.1s
[  4/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.2s
[  5/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.4s
[  6/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.1s
[  7/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.2s
[  8/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.4s
[  9/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.3s
[ 10/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.2s
[ 11/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.7s
[ 12/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 2.1s
[ 13/160] ✅ svc=frontend        ✅ type=ddos            | acc=100.0% | 3.1s
[ 14/160] ✅ svc=frontend 